# 🎙️ 企業 SOP 電影級配音工作站（Google Colab 免費 T4 GPU｜免設定一鍵版）

專為公司同仁設計：**零程式門檻、免費 T4 顯卡即可跑**，支援雙引擎自由切換：
- **BreezyVoice**（聯發科＋台大，Apache-2.0 可商用）：**台灣國語最道地**，支援注音精準糾音（如 `連假[:ㄐㄧㄚ4]`），需提供 6–10 秒參考乾聲。
- **VoxCPM2**（OpenBMB）：**48kHz 超高音質天花板**，支援聲音克隆，若未上傳參考音也可直接用文字描述「捏出指定台灣講師聲音」。

---
### 三步驟完成配音（一般同仁只要做這 3 步）
1. **確認右上角連線為 `T4 GPU`**（點選單列 `執行階段` → `變更執行階段類型` → 選 `T4 GPU`）。
2. **在下方第 1 格選好引擎**（預設 `BreezyVoice`，未錄音可切 `VoxCPM2` 純文字設計聲音），接著點選單列 **`執行階段` → `全部執行`**（Shortcut: `Ctrl+F9`）。
3. **第 3 格跳出上傳按鈕時**，選取你的 **`ref.wav`（6–10 秒安靜環境錄音）** 與 **`bpm_voice_jobs.json`**（若沒選 JSON 會自動套用內建 BPM 腳本）；全部跑完會**自動下載 `cloned_audio_ready.zip`**，解壓把裡面 `final_audio/` 的 `.mp3` 放回 `blender_3d_demo/bpm_sop/audio/` 即可秒速合成電影版影片！

In [ ]:
#@title ⚙️ 1. 選擇配音引擎與參數（一般人員只需選這裡）
ENGINE = "BreezyVoice" #@param ["BreezyVoice", "VoxCPM2"]
SEEDS_PER_SHOT = 3 #@param {type:"slider", min:1, max:6, step:1}
AUTO_PICK_WITH_WHISPER = True #@param {type:"boolean"}
WHISPER_MODEL = "small" #@param ["base", "small", "medium"]
REF_TRANSCRIPT = "" #@param {type:"string"}
VOXCPM2_VOICE_DESIGN = "一位親切專業的台灣職場講師，聲音自然清亮、吐字清晰、節奏明快" #@param {type:"string"}
VOXCPM2_STEPS = 10 #@param {type:"slider", min:5, max:20, step:1}
VOXCPM2_CFG = 2.0 #@param {type:"number"}
EXPORT_FORMAT = "mp3" #@param ["mp3", "wav"]
USE_GOOGLE_DRIVE_CACHE = False #@param {type:"boolean"}

import subprocess
from pathlib import Path

WORK_DIR = Path("/content/work")
WORK_DIR.mkdir(parents=True, exist_ok=True)

res = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
if res.returncode == 0 and res.stdout.strip():
    print("✅ 偵測到 GPU：", res.stdout.strip())
else:
    print("⚠️ 目前未偵測到 GPU！請點上方選單「執行階段 → 變更執行階段類型 → T4 GPU」後再執行。")
print(f"🎯 已選定引擎：{ENGINE} | 每句抽卡數：{SEEDS_PER_SHOT} | 自動驗字挑音：{AUTO_PICK_WITH_WHISPER} ({WHISPER_MODEL})")


In [ ]:
#@title 📦 2. 全自動安裝環境與下載模型（免手動設定，約 1.5–3 分鐘）
import os
import shutil
import subprocess
import sys
from pathlib import Path

def sh(cmd, check=True, env=None):
    merged = {**os.environ, **(env or {})}
    r = subprocess.run(cmd, shell=isinstance(cmd, str), text=True, capture_output=True, env=merged)
    if r.returncode != 0:
        print((r.stdout or "")[-1500:])
        print((r.stderr or "")[-1500:])
        if check:
            raise RuntimeError(f"指令失敗 (exit={r.returncode}): {cmd}")
    return r

# 0. 選擇性掛載 Google Drive 快取模型，下次免重下載
MODEL_CACHE = Path("/content/model_cache")
if USE_GOOGLE_DRIVE_CACHE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        MODEL_CACHE = Path("/content/drive/MyDrive/AI_Voice_Cache")
        print("💾 已啟用 Google Drive 模型快取：", MODEL_CACHE)
    except Exception as exc:
        print(f"⚠️ 掛載 Google Drive 失敗，改用本機暫存 ({exc})")
MODEL_CACHE.mkdir(parents=True, exist_ok=True)

# 1. 安裝系統基礎套件（uv、ffmpeg、挑音所需輕量工具）
if shutil.which("uv") is None:
    print("⏳ 安裝 uv 套件加速器...")
    sh([sys.executable, "-m", "pip", "install", "-q", "uv"])

if shutil.which("ffmpeg") is None:
    sh("apt-get -qq update && apt-get -qq install -y ffmpeg")

sh([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub", "faster-whisper", "opencc-python-reimplemented", "soundfile", "librosa"])

def install_torch_with_fallback(py_exe, pinned_version=None):
    for cu in ("cu128", "cu126", "cu124"):
        pkgs = [f"torch=={pinned_version}", f"torchaudio=={pinned_version}"] if pinned_version else ["torch", "torchaudio"]
        cmd = ["uv", "pip", "install", "-q", "--python", str(py_exe), *pkgs, "--index-url", f"https://download.pytorch.org/whl/{cu}"]
        r = sh(cmd, check=False)
        if r.returncode == 0:
            chk = sh([str(py_exe), "-c", "import torch; assert torch.cuda.is_available(); print(torch.__version__, torch.cuda.get_device_name(0))"], check=False)
            if chk.returncode == 0:
                print(f"✅ PyTorch CUDA ({cu}) 就緒：", chk.stdout.strip())
                return
    raise RuntimeError("無法安裝可用的 CUDA PyTorch，請確認目前執行階段為 GPU。")

if ENGINE == "BreezyVoice":
    BV_ROOT = Path("/content/BreezyVoice")
    BV_CODE = BV_ROOT / "code"
    BV_STUB = BV_ROOT / "winstub"
    BV_VENV = BV_ROOT / "venv"
    BV_PY = BV_VENV / "bin" / "python"
    BV_MODEL = MODEL_CACHE / "BreezyVoice-300M"

    if not (BV_CODE / "single_inference.py").exists():
        print("⏳ 下載 BreezyVoice 原始碼...")
        sh(["git", "clone", "-q", "--recurse-submodules", "https://github.com/mtkresearch/BreezyVoice", str(BV_CODE)])
        sh(["git", "-C", str(BV_CODE), "checkout", "-q", "d592c9d3e8927a0f53f68616387060dcd32a05ea"])
        sh(["git", "-C", str(BV_CODE), "submodule", "update", "-q", "--init", "--recursive"])

    # 建立免 pynini 直通文字正規化替身 (winstub)
    for rel, body in {
        "tn/__init__.py": "",
        "tn/chinese/__init__.py": "",
        "tn/chinese/normalizer.py": "class Normalizer:\n    def __init__(self, *a, **k): pass\n    def normalize(self, text): return text\n",
        "tn/english/__init__.py": "",
        "tn/english/normalizer.py": "class Normalizer:\n    def __init__(self, *a, **k): pass\n    def normalize(self, text): return text\n",
    }.items():
        p = BV_STUB / rel
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_text(body, encoding="utf-8")

    if not BV_PY.exists():
        print("⏳ 建立隔離 Python 3.10 環境並安裝 BreezyVoice 鎖定依賴...")
        sh(["uv", "venv", "-q", "-p", "3.10", str(BV_VENV)])
        sh(["uv", "pip", "install", "-q", "--python", str(BV_PY), "setuptools<70", "wheel"])
        install_torch_with_fallback(BV_PY, pinned_version="2.7.1")
        bv_req = [
            "absl-py==2.5.0", "aiohappyeyeballs==2.7.1", "aiohttp==3.14.4", "aiosignal==1.4.0",
            "annotated-types==0.8.0", "antlr4-python3-runtime==4.9.3", "anyio==4.15.1", "async-timeout==5.0.1",
            "attrs==26.1.0", "audioread==3.1.0", "beautifulsoup4==4.15.0", "certifi", "cffi",
            "charset-normalizer", "click", "cloudpickle==3.1.2", "coloredlogs==15.0.1", "conformer==0.3.2",
            "contourpy", "cryptography", "cycler", "datasets", "decorator==5.3.1", "diffusers==0.32.0",
            "dill", "einops==0.8.2", "exceptiongroup", "filelock", "flatbuffers", "fonttools",
            "frozenlist", "fsspec==2024.12.0", "g2pw==0.1.1", "gdown==5.1.0", "google-auth",
            "google-auth-oauthlib", "grpcio", "grpcio-tools", "h11", "huggingface-hub==0.36.2",
            "humanfriendly", "hydra-core==1.3.2", "hyperpyyaml==1.2.2", "idna", "importlib-metadata",
            "inflect==7.3.1", "jinja2", "joblib", "kiwisolver", "lazy-loader", "librosa==0.10.2",
            "lightning==2.2.4", "lightning-utilities", "llvmlite", "markdown", "markdown-it-py",
            "markupsafe", "matplotlib==3.7.5", "mdurl", "more-itertools", "mpmath", "msgpack",
            "multidict", "multiprocess", "networkx==3.1", "numba", "numpy==1.26.4", "oauthlib",
            "omegaconf==2.3.0", "onnxruntime-gpu", "openai-whisper==20231117",
            "opencc-python-reimplemented==0.1.7", "packaging==24.2", "pandas", "pillow",
            "platformdirs", "pooch", "propcache", "protobuf==4.25.0", "pyarrow", "pyasn1",
            "pyasn1-modules", "pycparser", "pydantic==2.7.0", "pydantic-core==2.18.1",
            "pydantic-settings==2.7.0", "pygments", "pyparsing", "pysocks", "python-dateutil",
            "python-dotenv", "pytorch-lightning", "pytz", "pyyaml", "regex", "requests",
            "requests-oauthlib", "rich==13.7.1", "ruamel-yaml==0.17.40", "ruamel-yaml-clib==0.2.15",
            "safetensors", "scikit-learn", "scipy", "six", "soundfile==0.12.1", "soupsieve",
            "soxr", "sympy", "tensorboard", "tensorboard-data-server", "threadpoolctl", "tiktoken",
            "tokenizers==0.20.3", "tomli", "torchmetrics", "tqdm", "transformers==4.46.3", "truststore", "typeguard",
            "typing-extensions", "tzdata", "urllib3", "werkzeug", "wget==3.2", "xxhash", "yarl", "zipp",
        ]
        req_file = Path("/content/req_bv.txt")
        req_file.write_text("\n".join(bv_req), encoding="utf-8")
        sh(["uv", "pip", "install", "-q", "--python", str(BV_PY), "--no-build-isolation", "--no-deps", "-r", str(req_file), "--index-strategy", "unsafe-best-match"])

    if not (BV_MODEL / "llm.pt").exists():
        print("⏳ 下載 MediaTek BreezyVoice-300M 權重（鎖定版本）...")
        from huggingface_hub import snapshot_download
        snapshot_download("MediaTek-Research/BreezyVoice-300M", revision="e33b502e0ac21c16b0ee0d00df66ac3fa737393d", local_dir=str(BV_MODEL))
    # 預載 G2PWModel（改由 HuggingFace Hub 300MB/s 直載，徹底避開 Google Drive 限流與境外阻擋）
    if not Path("/content/G2PWModel/g2pW.onnx").exists():
        print("⏳ 預載 G2PWModel (HuggingFace Hub alextomcat/G2PWModel)...")
        from huggingface_hub import snapshot_download
        snapshot_download("alextomcat/G2PWModel", local_dir="/content/G2PWModel")
    sh(f"ln -sfn /content/G2PWModel {BV_CODE}/G2PWModel")
    print("✅ BreezyVoice 環境與模型就緒！")

else:
    VX_ROOT = Path("/content/VoxCPM2")
    VX_VENV = VX_ROOT / "venv"
    VX_PY = VX_VENV / "bin" / "python"
    VX_MODEL = MODEL_CACHE / "VoxCPM2"

    if not VX_PY.exists():
        print("⏳ 建立隔離 Python 3.12 環境並安裝 VoxCPM2...")
        sh(["uv", "venv", "-q", "-p", "3.12", str(VX_VENV)])
        install_torch_with_fallback(VX_PY, pinned_version=None)
        sh(["uv", "pip", "install", "-q", "--python", str(VX_PY), "voxcpm==2.0.3", "soundfile", "huggingface_hub", "opencc-python-reimplemented"])

    if not (VX_MODEL / "model.safetensors").exists():
        print("⏳ 下載 OpenBMB VoxCPM2 48kHz 權重（約 4.9GB，Colab 機房約 40 秒）...")
        from huggingface_hub import snapshot_download
        snapshot_download("openbmb/VoxCPM2", revision="32279effe8c19989596f05d353d1447f51d9e915", local_dir=str(VX_MODEL))
    print("✅ VoxCPM2 環境與模型就緒！")


In [ ]:
#@title 🎙️ 3. 上傳參考聲音檔 (ref.wav) 與腳本 JSON（可略過直接用內建 BPM 腳本）
import gc
import json
import shutil
import subprocess
from pathlib import Path

WORK_DIR = Path("/content/work")
WORK_DIR.mkdir(parents=True, exist_ok=True)

DEFAULT_BPM_JOBS = {
    "project": "BPM_一般費用報銷單_私車公用",
    "default_engine": ENGINE,
    "seeds": SEEDS_PER_SHOT,
    "jobs": [
        {"name": "shot01", "text": "要報銷開車出差的費用，出發前先準備好三項實際佐證資料：第一是本人已經核准的出差登記單；第二是用 Google 地圖截下當次實際去程與回程的路線里程圖，每個人的距離都不同，依實際跑多少公里計算；第三則是遠通 一Tag 官方下載的通行費紀錄列印單，記得一定要有遠通電收的電子憑證專用章，不能只用線上即時截圖喔！"},
        {"name": "shot02", "text": "進到鼎新 B P M 系統後，點選 E P M 裡面的『一般費用報銷單』，先確認表頭的公司別、部門跟申請日期。接著到下方費用明細，點『出差登記單號』旁邊的查詢鈕，選擇你這趟實際出差的單號，不要選錯張，再核對帶入的日期、地點跟出差事由都與本人實際行程一致。"},
        {"name": "shot03", "text": "接下來新增私車公用里程費。請先注意右上角的『憑證日期』，記得一定要修改為這筆費用的實際發生或憑證日期，不要漏掉喔！接著費用類別選 A零五『國內出差私車公用』，金額依照你自己實際跑了幾公里、按公司核定單價計算後再填寫，稅別碼選 T零五 免稅；最後選實際費用地區，並在費用說明寫上本人車牌號碼後按『新增』。"},
        {"name": "shot04", "text": "這裡要特別注意，高速公路的 一Tag 通行費不能跟里程費算在同一筆！而且會計規定，通行費佐證必須下載含有『遠通電收電子憑證專用章』的官方紀錄列印單，不能只用網站上的未出帳截圖。確認本筆憑證日期後，依照單據上的實際扣款金額，另外新增一筆 A零七『國內出差交通 一Tag』，照實登打並與下方 A零五 分開列示。"},
        {"name": "shot05", "text": "接下來補充一般出差搭高鐵與停車費的實務填法，提醒大家，高鐵與停車費都不算在私車里程費裡面。先看高鐵 A零一，第一點記得先把右上角的憑證日期改成高鐵票實際日期；第二點是會計特別提醒，高鐵票的號碼格式跟一般發票不同，沒有前面兩碼英文字軌，在發票號碼欄請直接輸入高鐵車票上的 十三 碼數字票號。第三點看金額與稅別，高鐵通常選 T零二 應稅內含百分之五，這時報銷金額要直接輸入含稅總額，就像畫面範例 鍵入 六百五十 元，系統會自動算成未稅 六百一十九 加稅額 三十一 等於含稅 六百五十，請核對最下方三個金額欄位跟實際車票完全一致。"},
        {"name": "shot06", "text": "接著看停車費 A零六，一樣先把右上角憑證日期改成停車發票實際日期，中間依照實際發票填入賣方統編八碼與英文開頭十碼發票號碼。這裡幫大家整理會計最常提醒的金額 鍵入 法：如果選 T零一 應稅外加百分之五，像這張發票實付 一百六十 元，報銷金額欄就要 鍵入 未稅金額 一百五十二 元，系統會外加 八 元稅額，變成含稅 一百六十；相反地，如果選 T零二 應稅內含百分之五，報銷金額就要直接 鍵入 含稅總額；至於 T零五 或 T零四 免稅則直接填實付金額。不管哪一種，都一定要核對下方『原幣未稅加稅額等於原幣含稅』跟手中紙本發票完全一樣！"},
        {"name": "shot07", "text": "全部依照實際發生費用登打完之後，我們檢查一下底下的明細表，確認每一筆的憑證日期、費用類別、未稅與含稅金額都跟手中憑證一致，明細加起來跟表頭總金額一模一樣，也要把出差單、路線截圖跟相關明細全部上傳並確認檔案打得開。最後先按『儲存』，沒跳紅字錯誤再按『送出』，並把 B P M 單號記下來。"},
        {"name": "shot08", "text": "線上送出還不算完全結束喔！請把申請單印出來，連同實際發生的發票、高鐵票或停車收據正本、路線截圖跟費用計算依據整理好，先交給人事單位核對；等人事確認正本金額與日期跟系統登打完全一致之後，就會幫你轉交給會計做最後審核與撥款囉！"},
    ],
}

UPLOAD_DIR = WORK_DIR / "uploads"
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

print("📤 請選取要上傳的檔案（可同時多選：你的 6–10 秒乾淨人聲檔 .wav/.mp3/.m4a ＋ bpm_voice_jobs.json；若都不選直接按取消則會自動使用預設腳本）：")
uploaded_names = []
try:
    from google.colab import files
    uploaded = files.upload()
    for fname, raw_bytes in uploaded.items():
        dst = UPLOAD_DIR / Path(fname).name
        dst.write_bytes(raw_bytes)
        uploaded_names.append(dst)
        print("  ✔ 已收到：", dst.name)
except Exception as exc:
    print("ℹ️ 非 Colab 互動環境或略過上傳：", exc)

# 尋找上傳的 JSON 與參考音訊
json_candidates = [p for p in uploaded_names if p.suffix.lower() == ".json"] or list(UPLOAD_DIR.glob("*.json"))
JOBS_PATH = WORK_DIR / "jobs.json"
if json_candidates:
    shutil.copy2(json_candidates[0], JOBS_PATH)
    print("📄 已套用你上傳的腳本 JSON：", json_candidates[0].name)
else:
    JOBS_PATH.write_text(json.dumps(DEFAULT_BPM_JOBS, ensure_ascii=False, indent=2), encoding="utf-8")
    print("📄 未上傳 JSON，自動套用內建 BPM 私車公用 8 鏡台灣口音腳本。")

audio_exts = {".wav", ".mp3", ".m4a", ".ogg", ".flac"}
audio_candidates = [p for p in uploaded_names if p.suffix.lower() in audio_exts] or [p for p in UPLOAD_DIR.iterdir() if p.suffix.lower() in audio_exts]
REF_WAV = WORK_DIR / "ref_clean.wav"
if REF_WAV.exists():
    REF_WAV.unlink()

if audio_candidates:
    src_ref = audio_candidates[0]
    # 轉成單聲道 24kHz PCM16 並裁去頭尾超長靜音，提升克隆純淨度
    subprocess.run([
        "ffmpeg", "-y", "-i", str(src_ref),
        "-ac", "1", "-ar", "24000", "-sample_fmt", "s16",
        "-af", "silenceremove=start_periods=1:start_threshold=-45dB:detection=peak,areverse,silenceremove=start_periods=1:start_threshold=-45dB:detection=peak,areverse",
        str(REF_WAV),
    ], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print(f"🎧 參考聲音檔已標準化淨化：{src_ref.name} -> {REF_WAV.name}")
else:
    if ENGINE == "BreezyVoice":
        raise RuntimeError("⚠️ BreezyVoice 必須有 6–10 秒參考人聲檔 (.wav/.mp3/.m4a) 才能克隆！請重新執行本格上傳你的錄音，或將第 1 格 ENGINE 改選為 VoxCPM2（可免錄音直接用文字捏出聲音）。")
    print("✨ 未上傳參考人聲，將使用 VoxCPM2 聲音設計模式：", VOXCPM2_VOICE_DESIGN)

# 優先讀取 jobs.json 內的 ref_text；若未填再以 Whisper small (CPU) + librosa 陣列聽寫（避開 Python 3.13 PyAV metadata_errors 相容問題）
RESOLVED_REF_TEXT = REF_TRANSCRIPT.strip()
if not RESOLVED_REF_TEXT and JOBS_PATH.exists():
    try:
        RESOLVED_REF_TEXT = str(json.loads(JOBS_PATH.read_text(encoding="utf-8-sig")).get("ref_text", "")).strip()
    except Exception:
        pass
if ENGINE == "BreezyVoice" and not RESOLVED_REF_TEXT:
    print("⏳ 正在自動聽寫參考音逐字稿 (faster-whisper small CPU)...")
    import librosa, opencc
    from faster_whisper import WhisperModel
    wm = WhisperModel("small", device="cpu", compute_type="int8")
    wav16k, _ = librosa.load(str(REF_WAV), sr=16000, mono=True)
    segs, _ = wm.transcribe(wav16k.astype("float32"), language="zh", beam_size=5)
    raw_tr = "".join(s.text for s in segs).strip()
    RESOLVED_REF_TEXT = opencc.OpenCC("s2twp").convert(raw_tr)
    del wm
    gc.collect()
    print("📝 自動聽寫參考逐字稿：", RESOLVED_REF_TEXT)

(WORK_DIR / "resolved_ref_text.txt").write_text(RESOLVED_REF_TEXT, encoding="utf-8")


In [ ]:
#@title 🚀 4. T4 GPU 批次克隆配音（獨立子程序執行，防 12GB RAM 爆掉）
import gc
import os
import subprocess
from pathlib import Path

WORK_DIR = Path("/content/work")
CAND_DIR = WORK_DIR / "candidates"
CAND_DIR.mkdir(parents=True, exist_ok=True)

gc.collect()
WORKER_SCRIPT = WORK_DIR / "run_tts_worker.py"

if ENGINE == "BreezyVoice":
    worker_code = r'''import json, os, re, sys, time
from pathlib import Path
BV_CODE = Path("/content/BreezyVoice/code")
BV_STUB = Path("/content/BreezyVoice/winstub")
MODEL_DIR = Path(sys.argv[1])
JOBS_PATH = Path(sys.argv[2])
REF_WAV = Path(sys.argv[3])
REF_TEXT = Path(sys.argv[4]).read_text(encoding="utf-8").strip()
CAND_DIR = Path(sys.argv[5])
SEEDS = int(sys.argv[6])

os.chdir(BV_CODE)
for p in (BV_STUB, BV_CODE, BV_CODE / "third_party" / "Matcha-TTS"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import torch
import torchaudio
from g2pw import G2PWConverter
from single_inference import CustomCosyVoice, get_bopomofo_rare
from cosyvoice.utils.file_utils import load_wav

TW_BOPO = {
    "連假": "連假[:ㄐㄧㄚ4]", "放假": "放假[:ㄐㄧㄚ4]", "請假": "請假[:ㄐㄧㄚ4]",
    "暑假": "暑假[:ㄐㄧㄚ4]", "寒假": "寒假[:ㄐㄧㄚ4]", "假期": "假[:ㄐㄧㄚ4]期", "假日": "假[:ㄐㄧㄚ4]日",
}
DIG = "零一二三四五六七八九"
def int_zh(n):
    if n < 10: return DIG[n]
    if n < 100:
        t, o = divmod(n, 10)
        return ("" if t == 1 else DIG[t]) + "十" + (DIG[o] if o else "")
    if n < 1000:
        h, r = divmod(n, 100)
        return DIG[h] + "百" + ("" if r == 0 else ("零" + DIG[r] if r < 10 else ("一" + int_zh(r) if r < 20 else int_zh(r))))
    return "".join(DIG[int(c)] for c in str(n))

def prep(text):
    text = re.sub(r"\[:[^\]]*\]|\d+", lambda m: m.group() if m.group().startswith("[") else int_zh(int(m.group())), text)
    for k, v in TW_BOPO.items():
        if v not in text:
            text = text.replace(k, v)
    return text.replace("……", "，").replace("——", "，").replace("～", "！").strip()

t0 = time.time()
cv = CustomCosyVoice(str(MODEL_DIR))
conv = G2PWConverter()
conv.num_workers = 0
print(f"[BreezyVoice] 模型載入完成 ({time.time() - t0:.1f}s)", flush=True)

spec = json.loads(JOBS_PATH.read_text(encoding="utf-8-sig"))
ps = load_wav(str(REF_WAV), 16000)
pt = get_bopomofo_rare(cv.frontend.text_normalize_new(REF_TEXT, split=False), conv)

for j in spec["jobs"]:
    text = prep(j["text"])
    ct = get_bopomofo_rare(text if "[" in text else cv.frontend.text_normalize_new(text, split=False), conv)
    n_seeds = int(j.get("seeds", SEEDS))
    for seed in range(1, n_seeds + 1):
        dst = CAND_DIR / f"{j['name']}_s{seed}.wav"
        if dst.exists():
            continue
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        ts = time.time()
        out = cv.inference_zero_shot_no_normalize(ct, pt, ps)
        torchaudio.save(str(dst), out["tts_speech"], 22050)
        print(f"  ✔ {dst.name} ({time.time() - ts:.2f}s)", flush=True)
print(f"[BreezyVoice] 全部完成！總耗時 {time.time() - t0:.1f}s", flush=True)
'''
    WORKER_SCRIPT.write_text(worker_code, encoding="utf-8")
    cmd = [
        "/content/BreezyVoice/venv/bin/python",
        str(WORKER_SCRIPT),
        str(MODEL_CACHE / "BreezyVoice-300M"),
        str(WORK_DIR / "jobs.json"),
        str(WORK_DIR / "ref_clean.wav"),
        str(WORK_DIR / "resolved_ref_text.txt"),
        str(CAND_DIR),
        str(SEEDS_PER_SHOT),
    ]
else:
    worker_code = r'''import json, re, sys, time
from pathlib import Path
import soundfile as sf
import torch
from voxcpm import VoxCPM

MODEL_DIR = Path(sys.argv[1])
JOBS_PATH = Path(sys.argv[2])
REF_WAV = Path(sys.argv[3])
CAND_DIR = Path(sys.argv[4])
SEEDS = int(sys.argv[5])
VOICE_DESIGN = sys.argv[6]
STEPS = int(sys.argv[7])
CFG = float(sys.argv[8])

t0 = time.time()
model = VoxCPM.from_pretrained(str(MODEL_DIR), load_denoiser=False, optimize=False, device="cuda")
print(f"[VoxCPM2] 模型載入完成 ({time.time() - t0:.1f}s)", flush=True)

spec = json.loads(JOBS_PATH.read_text(encoding="utf-8-sig"))
has_ref = REF_WAV.exists()
design_str = (spec.get("voice_design") or VOICE_DESIGN).strip()

for j in spec["jobs"]:
    clean_txt = re.sub(r"\[:[^\]]*\]", "", j["text"]).strip()
    prompt_txt = clean_txt if has_ref else f"({design_str}){clean_txt}"
    n_seeds = int(j.get("seeds", SEEDS))
    for seed in range(1, n_seeds + 1):
        dst = CAND_DIR / f"{j['name']}_s{seed}.wav"
        if dst.exists():
            continue
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        kwargs = {
            "text": prompt_txt,
            "cfg_value": CFG,
            "inference_timesteps": STEPS,
            "normalize": False,
            "denoise": False,
        }
        if has_ref:
            kwargs["reference_wav_path"] = str(REF_WAV)
        ts = time.time()
        wav = model.generate(**kwargs)
        sf.write(str(dst), wav, model.tts_model.sample_rate, subtype="PCM_16")
        print(f"  ✔ {dst.name} ({time.time() - ts:.2f}s)", flush=True)
print(f"[VoxCPM2] 全部完成！總耗時 {time.time() - t0:.1f}s", flush=True)
'''
    WORKER_SCRIPT.write_text(worker_code, encoding="utf-8")
    cmd = [
        "/content/VoxCPM2/venv/bin/python",
        str(WORKER_SCRIPT),
        str(MODEL_CACHE / "VoxCPM2"),
        str(WORK_DIR / "jobs.json"),
        str(WORK_DIR / "ref_clean.wav"),
        str(CAND_DIR),
        str(SEEDS_PER_SHOT),
        VOXCPM2_VOICE_DESIGN,
        str(VOXCPM2_STEPS),
        str(VOXCPM2_CFG),
    ]

print(f"🚀 啟動獨立子程序進行 {ENGINE} T4 GPU 批次配音...")
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
for line in proc.stdout:
    print(line, end="")
rc = proc.wait()
if rc != 0:
    raise RuntimeError(f"配音子程序結束碼異常：{rc}")
print("🧹 子程序已結束，T4 顯存與 12GB 系統記憶體已 100% 釋放！")


In [ ]:
#@title 🏆 5. Whisper 自動驗字挑音＋去靜音正規化＋試聽＋打包下載 ZIP
import difflib
import gc
import json
import re
import shutil
import subprocess
import zipfile
from pathlib import Path

WORK_DIR = Path("/content/work")
CAND_DIR = WORK_DIR / "candidates"
FINAL_DIR = WORK_DIR / "final_audio"
FINAL_DIR.mkdir(parents=True, exist_ok=True)

spec = json.loads((WORK_DIR / "jobs.json").read_text(encoding="utf-8-sig"))

def clean_chars(s):
    s = re.sub(r"\[:[^\]]*\]", "", s)
    return re.sub(r"[^\u4e00-\u9fffA-Za-z0-9]", "", s).upper()

asr_model = None
s2t = None
if AUTO_PICK_WITH_WHISPER:
    import opencc
    from faster_whisper import WhisperModel
    s2t = opencc.OpenCC("s2twp")
    try:
        asr_model = WhisperModel(WHISPER_MODEL, device="cuda", compute_type="int8")
        print(f"🔍 已載入 Whisper {WHISPER_MODEL} (GPU int8) 進行自動驗字挑音...")
    except Exception:
        asr_model = WhisperModel(WHISPER_MODEL, device="cpu", compute_type="int8")
        print(f"🔍 已載入 Whisper {WHISPER_MODEL} (CPU int8) 進行自動驗字挑音...")

import librosa
import soundfile as sf

report = {}
for j in spec["jobs"]:
    name = j["name"]
    target_clean = clean_chars(j["text"])
    cands = sorted(CAND_DIR.glob(f"{name}_s*.wav"), key=lambda p: int(p.stem.rsplit("_s", 1)[1]))
    if not cands:
        print(f"⚠️ 找不到 {name} 候選音檔，略過")
        continue

    scored = []
    for c in cands:
        x, sr = sf.read(str(c), dtype="float32")
        mono = x.mean(axis=1) if x.ndim > 1 else x
        trimmed, _ = librosa.effects.trim(mono, top_db=38)
        dur_s = round(max(len(trimmed), 1) / sr, 2)
        sim = 1.0
        heard = ""
        if asr_model is not None:
            wav16k = librosa.resample(mono, orig_sr=sr, target_sr=16000).astype("float32") if sr != 16000 else mono.astype("float32")
            segs, _ = asr_model.transcribe(wav16k, language="zh", beam_size=5)
            heard = s2t.convert("".join(s.text for s in segs)).strip()
            sim = round(difflib.SequenceMatcher(None, clean_chars(heard), target_clean).ratio(), 4)
        scored.append({"file": c.name, "sim": sim, "dur_s": dur_s, "heard": heard})

    # 過濾掉明顯過短/爆長異常檔後，取字元相似度最高者
    valid = [r for r in scored if r["dur_s"] >= 0.8] or scored
    best = max(valid, key=lambda r: (r["sim"], -abs(r["dur_s"] - len(target_clean) * 0.24)))
    best_src = CAND_DIR / best["file"]

    # 切除前後冗餘靜音 (保留開頭 80ms / 結尾 150ms) 並做響度標準化 (-16 LUFS)
    x, sr = sf.read(str(best_src), dtype="float32")
    mono = x.mean(axis=1) if x.ndim > 1 else x
    _, (a, b) = librosa.effects.trim(mono, top_db=40)
    a = max(0, a - int(0.08 * sr))
    b = min(len(x), b + int(0.15 * sr))
    trimmed_wav = WORK_DIR / f"{name}_trimmed.wav"
    sf.write(str(trimmed_wav), x[a:b], sr, subtype="PCM_16")

    ext = ".mp3" if EXPORT_FORMAT == "mp3" else ".wav"
    final_out = FINAL_DIR / f"{name}{ext}"
    ff_cmd = ["ffmpeg", "-y", "-i", str(trimmed_wav), "-af", "loudnorm=I=-16:TP=-1.5:LRA=11", "-ar", "44100", "-ac", "2"]
    if ext == ".mp3":
        ff_cmd += ["-b:a", "192k"]
    ff_cmd.append(str(final_out))
    subprocess.run(ff_cmd, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    trimmed_wav.unlink(missing_ok=True)

    report[name] = {"picked": best["file"], "sim": best["sim"], "dur_s": best["dur_s"], "candidates": scored}
    print(f"✅ {name} -> 選用 {best['file']} (字元對齊度 {best['sim']*100:.1f}%, {best['dur_s']}s) -> {final_out.name}")

if asr_model is not None:
    del asr_model
    gc.collect()

REPORT_PATH = WORK_DIR / "pick_report.json"
REPORT_PATH.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")

ZIP_PATH = Path("/content/cloned_audio_ready.zip")
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(REPORT_PATH, "pick_report.json")
    for f in sorted(FINAL_DIR.iterdir()):
        zf.write(f, f"final_audio/{f.name}")
    for f in sorted(CAND_DIR.iterdir()):
        zf.write(f, f"candidates/{f.name}")
print(f"\n📦 已打包：{ZIP_PATH} ({ZIP_PATH.stat().st_size / 1024:.1f} KB)")

# 線上試聽與自動下載
try:
    from IPython.display import Audio, HTML, display
    display(HTML("<h4>🎧 成品立即試聽（挑不滿意也可打開壓縮檔裡的 candidates/ 換別顆 Seed）：</h4>"))
    for f in sorted(FINAL_DIR.iterdir()):
        display(HTML(f"<b>{f.name}</b>"))
        display(Audio(filename=str(f)))
except Exception:
    pass

try:
    from google.colab import files
    files.download(str(ZIP_PATH))
    print("⬇ 已觸發瀏覽器自動下載 cloned_audio_ready.zip！解壓後把 final_audio/ 裡的檔案放入 blender_3d_demo/bpm_sop/audio/ 即可。")
except Exception:
    print(f"💡 成品位置：{ZIP_PATH}")
